# Movie / OTT Recommendation & Trend Analysis
## Data Mining & Analysis Mini Project (CO1 to CO5 Pipeline Demonstration)

This notebook demonstrates every module of the project end-to-end for viva and evaluation:
- **CO1**: Fact Constellation Data Warehouse & OLAP Queries
- **CO2**: Apriori Association Rules Mining
- **CO2**: Naive Bayes Genre Classification from Plot Summaries
- **CO4**: Decision Tree & Random Forest Movie Rating Class Prediction
- **CO3 & CO4**: Linear, Ridge, Lasso & Decision Tree Regression
- **CO5**: Time Series Stationarity Analysis (ADF) & ARIMA Projections
- **CO5**: Content, Collaborative, Rule-Based & Hybrid Recommendation

### 1. Setup & Imports

In [ ]:
import sqlite3
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set random seeds for reproducibility
np.random.seed(42)
pd.set_option('display.max_columns', 15)
print("Environment initialized!")

### 2. Module A (CO1): Fact Constellation Warehouse & OLAP Analysis

In [ ]:
conn = sqlite3.connect('../warehouse.db')

# Audit row counts across the fact constellation schema
tables = ['dim_date', 'dim_user', 'dim_movie', 'dim_genre', 'fact_ratings', 'dim_title', 'fact_titles']
summary = []
for t in tables:
    cnt = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    summary.append({'Table': t, 'Row Count': cnt})

pd.DataFrame(summary)

In [ ]:
# OLAP Query: Top Genres by total rating count and average rating
olap_q = """
SELECT g.genre_name, COUNT(f.rating_id) as total_ratings, ROUND(AVG(f.rating), 3) as avg_rating
FROM fact_ratings f
JOIN movie_genre mg ON f.movie_id = mg.movie_id
JOIN dim_genre g ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY total_ratings DESC
LIMIT 8;
"""
pd.read_sql_query(olap_q, conn)

### 3. Module B (CO2): Apriori Association Rules Mining

In [ ]:
from mining.apriori_rules import get_rules_for_movie

rules_payload = joblib.load('../models/rules.joblib')
print(f"Loaded {len(rules_payload['strict_rules'])} strict rules and {len(rules_payload['lenient_rules'])} lenient rules.")

# Query top association rules for 'Kill Bill: Vol. 1' or 'The Matrix'
matrix_id = 2571  # The Matrix (1999)
matrix_rules = get_rules_for_movie(matrix_id, k=5, rules_payload=rules_payload)
pd.DataFrame(matrix_rules)[['antecedent_str', 'consequent_str', 'support', 'confidence', 'lift']]

### 4. Module C1 (CO2): Naive Bayes Plot Genre Classification

In [ ]:
from mining.classifier import predict_genre_from_text

sample_plot = "A fearless space crew embarks on an interstellar expedition to explore a mysterious wormhole near Saturn."
preds = predict_genre_from_text(sample_plot, k=3, models_dir=Path('../models'))
print(f"Plot: {sample_plot}\n")
pd.DataFrame(preds)

### 5. Module C2 & D (CO3, CO4): Rating Classification & Regression

In [ ]:
from mining.classifier import predict_movie_rating_class
from mining.regression import predict_movie_rating

# Predict for a high-profile Sci-Fi / Action movie
genres = ['Action', 'Sci-Fi', 'Thriller']
year = 2022
n_ratings = 150
n_tags = 6

cls_res = predict_movie_rating_class(genres, year, n_ratings, n_tags, models_dir=Path('../models'))
reg_score = predict_movie_rating(genres, year, n_ratings, n_tags, models_dir=Path('../models'))

print("=== PREDICTION RESULTS ===")
print(f"Predicted Rating Class (Random Forest): {cls_res['predicted_class']}")
print(f"Class Probabilities: {cls_res['probabilities']}")
print(f"Predicted Continuous Rating (Lasso Regression): ★ {reg_score} / 5.0")

### 6. Module E (CO5): ARIMA Time Series Trends & Forecast

In [ ]:
arima_data = joblib.load('../models/arima_models.joblib')
nf_entry = arima_data['netflix']

print(f"Netflix Model Order: ARIMA{nf_entry['order']}")
print(f"12-Month Out-of-Sample Test RMSE: {nf_entry['metrics']['arima_rmse']} (vs Naive: {nf_entry['metrics']['naive_rmse']})")
print(f"Differenced ADF p-value: {nf_entry['stationarity']['differenced_d1']['p_value']:.4e}")

# Display first 6 forecast months
nf_entry['forecast_df'].head(6)

### 7. Module F (CO5): Personalized Hybrid Recommendations

In [ ]:
from mining.recommender import recommend

# Query recommendations for 'Toy Story (1995)' [movie_id = 1]
recs = recommend(movie_id=1, k=5, method='hybrid', models_dir=Path('../models'))
pd.DataFrame(recs)[['title', 'year', 'score', 'why']]

In [ ]:
conn.close()
print("Demonstration pipeline finished successfully!")